In [11]:
import fitz

def extract_text_from_pdf(pdf_path: str) -> list[dict]:
    try:
        doc = fitz.open(pdf_path)
        pages = []
        for page_num in range(len(doc)):
            page = doc[page_num]
            text = page.get_text("text")
            pages.append({
                "page_number": page_num+1,
                "content": text,
                "metadata": {"source": pdf_path, "page": page_num+1}
            })
    except Exception as e:
        print(f"Got Exception: {e}")
    finally:
        doc.close()
    return pages

In [23]:
from path import Path

file_path = input("Enter pdf path: ")
if not file_path:
    file_path = "../PostgreSQL_Complete_Notes.pdf"
path = Path(file_path)

if not path.exists():
    raise FileNotFoundError(f"File Not present in: {file_path}")

pages = extract_text_from_pdf(path)

In [25]:
# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_document(pages: list[dict], chunk_size=800, chunk_overlap=150):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=['\n\n','\n', '.', ' ', ''],
        length_function=len
    )
    chunks = []
    for page in pages:
        splits = text_splitter.split_text(page["content"])
        for i, split in enumerate(splits):
            chunks.append({
                'content': split,
                'metadata': {
                    **page["metadata"],
                    "chunk_id": i, 
                    "chunk_size": len(split)
                }
            })
    return chunks

In [26]:
chunks = chunk_document(pages)

In [27]:
len(chunks)

81

In [29]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10662.26it/s]


In [31]:
def generate_embeddings(chunks: list[dict]):
    texts = [chunk['content'] for chunk in chunks]
    embeddings = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True)
    for chunk, emb in zip(chunks,embeddings):
        chunk["embedding"] = emb.tolist()
    return chunks

In [32]:
embedded_chunks = generate_embeddings(chunks)

Batches: 100%|██████████| 3/3 [00:02<00:00,  1.42it/s]


In [33]:
## Storing in elastic search

from elasticsearch import Elasticsearch, helpers

es = Elasticsearch('http://localhost:9200')
es.info()

ObjectApiResponse({'name': '473fbd3889bc', 'cluster_name': 'docker-cluster', 'cluster_uuid': '7KOoc0V7Sh2FPelmVBu6Sw', 'version': {'number': '8.13.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '09df99393193b2c53d92899662a8b8b3c55b45cd', 'build_date': '2024-03-22T03:35:46.757803203Z', 'build_snapshot': False, 'lucene_version': '9.10.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [34]:
index = 'rag_pdf'
mapping = {
    "settings": {
        'number_of_shards':1,
        "number_of_replicas":0
    },
    "mappings": {
        "properties": {
            "content": {'type': 'text'},
            'embedding': {
                'type': 'dense_vector',
                'dims': 384,
                'index': True,
                'similarity': 'cosine'
            },
            'metadata': {'type': 'object'}
        }
    }
}

if not es.indices.exists(index=index):
    es.indices.create(index=index, body=mapping)

In [35]:
# Bulk Index
import uuid
def bulk_index_chunks(chunks):
    actions= []
    for chunk in chunks:
        actions.append({
            "_index": index,
            "_id": str(uuid.uuid4()),
            "_source": {
                'content': chunk['content'],
                'embedding': chunk['embedding'],
                'metadata': chunk['metadata']
            }
        })
    helpers.bulk(es, actions)

bulk_index_chunks(embedded_chunks)
print("Indexing Completed!")

Indexing Completed!


In [36]:
def retrieve_relevant_chunks(query: str, top_k = 5):
    query_embedding = embedder.encode(query, normalize_embeddings=True).tolist()

    knn_query = {
        "knn": {
            "field": "embedding",
            "query_vector": query_embedding,
            "k": top_k,
            "num_candidates":50
        },
        "_source": ["content", "metadata"]
    }

    response = es.search(index=index, body=knn_query)

    results = []
    for hit in response['hits']['hits']:
        results.append({
            "score": hit["_score"],
            "content": hit['_source']["content"],
            "metadata": hit["_source"]["metadata"]
        })
    return results

In [37]:
relevant = retrieve_relevant_chunks("how to install postgres")

for r in relevant:
    print(f"Score: {r["score"]:.4f} | {r['content'][:300]}...")

Score: 0.8053 | PostgreSQL — Complete Notes
From Beginner to Advanced
What is PostgreSQL?
PostgreSQL (also called Postgres) is a powerful, open-source Relational Database Management
System (RDBMS).
It is based on SQL (Structured Query Language).
It was earlier known as POSTGRES, developed at UC Berkeley.
PostgreSQL...
Score: 0.7972 | Step 4 — Start / Stop / Restart
Step 5 — Access PostgreSQL shell
Or in one command:
Step 6 — Exit the shell
Step 7 — Set password for postgres user
PostgreSQL vs MySQL — Key Differences
Feature
MySQL
PostgreSQL
Type
Relational
Object-Relational
Compliance
Partial SQL
Full SQL Standard
JSON Support
B...
Score: 0.6792 | ROLES & USERS in PostgreSQL :-
In PostgreSQL, users and roles are the same thing. A role can have login privilege.
-- Create a schema
CREATE SCHEMA hr;
-- Create table in schema
CREATE TABLE hr.employees (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100)
);
-- Query from schema
SELECT * FROM hr.empl...
Score: 0.6763 | Feature
MySQL
PostgreSQL